# 🛡️ Phase 2 — Data Preprocessing & Leakage Prevention Pipeline

**F.R.A.U.D.S.H.I.E.L.D.** (*Financial Risk Assessment & Unauthorized Data Shield*)

---

## 🎯 Objective
Raw financial transaction data cannot be fed directly into machine learning algorithms:
1. **String identifiers** (`nameOrig`, `nameDest`) have high cardinality and cause catastrophic overfitting if not handled.
2. **Transaction types** (`PAYMENT`, `TRANSFER`, etc.) are categorical and require mathematical encoding.
3. **Financial amounts** have extreme right-skewness and heavy tails, requiring robust scaling rather than standard z-score normalization.
4. **Data Leakage Risk**: If transformations (scalers, encoders) are fit on the full dataset before splitting, information from the test set leaks into the training pipeline, invalidating all evaluation results.

### Core Phase 2 Deliverables:
- Domain-specific feature engineering (balance discrepancy calculations).
- Strict train/test splitting before any transformer fits.
- Comparison of **Stratified Split** vs. **Time-Based Split**.
- Categorical encoding via `OneHotEncoder(handle_unknown='ignore')`.
- Numerical scaling via `RobustScaler`.
- Modular, production-ready `FraudDataPreprocessor` pipeline.

## 1. Setup & Environment Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, MODELS_DIR
from src.data.loader import find_paysim_file, load_dataset
from src.data.preprocessor import FraudDataPreprocessor, split_data

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Environment initialized successfully.")

## 2. Load Transaction Data
We load either the full PaySim CSV from `data/raw/` or a representative schema sample if running in development mode.

In [ ]:
dataset_file = find_paysim_file()
if dataset_file and dataset_file.exists():
    print(f"Loading dataset from {dataset_file}...")
    # For fast and memory-friendly interactive execution, load up to 250,000 rows
    df = load_dataset(dataset_file, nrows=250000, use_optimized_dtypes=True)
else:
    print("Using representative PaySim schema sample for pipeline verification...")
    np.random.seed(42)
    n = 100000
    types = np.random.choice(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'], size=n, p=[0.34, 0.08, 0.35, 0.01, 0.22])
    amounts = np.random.exponential(scale=150000, size=n).astype(np.float32)
    old_bal_org = np.random.exponential(scale=800000, size=n).astype(np.float32)
    new_bal_org = np.maximum(0, old_bal_org - amounts)
    old_bal_dest = np.random.exponential(scale=1000000, size=n).astype(np.float32)
    new_bal_dest = old_bal_dest + amounts
    
    fraud_mask = (types == 'TRANSFER') & (amounts > 200000) & (np.random.rand(n) < 0.05)
    fraud_mask |= (types == 'CASH_OUT') & (amounts > 150000) & (np.random.rand(n) < 0.03)
    is_fraud = fraud_mask.astype(np.uint8)
    is_flagged = ((amounts > 200000) & (is_fraud == 1) & (np.random.rand(n) < 0.01)).astype(np.uint8)
    
    df = pd.DataFrame({
        'step': np.sort(np.random.randint(1, 745, size=n, dtype=np.uint16)),
        'type': pd.Categorical(types),
        'amount': amounts,
        'nameOrig': [f'C{i:09d}' for i in range(n)],
        'oldbalanceOrg': old_bal_org,
        'newbalanceOrig': new_bal_org,
        'nameDest': [f'M{i:09d}' if t == 'PAYMENT' else f'C{i+50000:09d}' for i, t in enumerate(types)],
        'oldbalanceDest': old_bal_dest,
        'newbalanceDest': new_bal_dest,
        'isFraud': is_fraud,
        'isFlaggedFraud': is_flagged
    })

print(f"Loaded DataFrame shape: {df.shape[0]:,} rows x {df.shape[1]} columns.")

## 3. Investigating String Identifiers & High Cardinality
`nameOrig` (originating customer) and `nameDest` (recipient) are unique alphanumeric strings (e.g. `C123456789`).
Let's see how many unique identifiers exist.

In [ ]:
unique_orig = df['nameOrig'].nunique()
unique_dest = df['nameDest'].nunique()
print(f"Total rows: {len(df):,}")
print(f"Unique origin accounts (nameOrig): {unique_orig:,} ({unique_orig / len(df) * 100:.1f}%)")
print(f"Unique destination accounts (nameDest): {unique_dest:,} ({unique_dest / len(df) * 100:.1f}%)")

> 🧠 **Machine Learning Lesson:**
> Because virtually every transaction originates from a distinct ID, feeding raw IDs into a model causes severe **cardinality explosion** (millions of dummy columns) or allows the model to memorize specific IDs instead of generalized fraud patterns.
> **Decision:** We drop the raw `nameOrig` and `nameDest` IDs, but first extract whether the recipient is a **Merchant** (`M...`) or a customer (`C...`), stored in `isDestMerchant`.

## 4. Feature Engineering: Balance Discrepancy Errors
In legitimate accounting:
- $\text{newbalanceOrig} = \text{oldbalanceOrg} - \text{amount} \implies \text{errorBalanceOrig} = \text{newbalanceOrig} + \text{amount} - \text{oldbalanceOrg} \approx 0$
- $\text{newbalanceDest} = \text{oldbalanceDest} + \text{amount} \implies \text{errorBalanceDest} = \text{oldbalanceDest} + \text{amount} - \text{newbalanceDest} \approx 0$

Fraudulent transactions frequently have large non-zero discrepancy errors (e.g. account emptied to 0, destination balance unaffected, or artificial zero balances).
Let's visualize the discrepancy between legitimate and fraudulent transactions.

In [ ]:
err_orig = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
err_dest = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

print("Balance Error Discrepancy Summary by Fraud Status:")
temp_df = pd.DataFrame({
    'isFraud': df['isFraud'],
    'errorBalanceOrig': err_orig,
    'errorBalanceDest': err_dest
})
display(temp_df.groupby('isFraud').mean())

Notice how fraudulent transactions have massively divergent balance errors! These two features provide strong signals to tree-based and linear models.

## 5. Preventing Data Leakage: Split Before Fitting

### What is Data Leakage?
If we calculate the mean, standard deviation, or median across the **entire dataset** and use it to scale features before splitting, the training set will contain information about the test set distribution.

```
❌ INCORRECT (LEAKAGE):
   [ Full Dataset ] --> Fit Scaler & Encoders --> Split into Train / Test
   (Result: Test statistics contaminate training. Model performs poorly in real world.)

✅ CORRECT (LEAKAGE-FREE):
   [ Full Dataset ] --> Split into Train / Test
                           |
                           +--> Fit Scaler on Train ONLY --> Transform Train
                           +--> Transform Test using Train's fitted parameters
```

In [ ]:
# Stratified Train / Test Split
X_train, X_test, y_train, y_test = split_data(
    df, target_col="isFraud", test_size=0.20, split_strategy="stratified", random_state=42
)

print(f"X_train shape: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
print(f"X_test shape:  {X_test.shape[0]:,} rows x {X_test.shape[1]} columns\n")

print(f"Train Fraud Rate: {y_train.mean() * 100:.4f}% ({y_train.sum():,} fraud cases)")
print(f"Test Fraud Rate:  {y_test.mean() * 100:.4f}% ({y_test.sum():,} fraud cases)")

## 6. Time-Based Split: A Realistic Alternative
In production banking, a model trained on past transactions must predict **future transactions**.
The `step` feature maps to hours across 30 days. Let's inspect a time-based split.

In [ ]:
X_tr_t, X_te_t, y_tr_t, y_te_t = split_data(
    df, target_col="isFraud", test_size=0.20, split_strategy="time_based"
)
print(f"Time-based Train Step Range: [{X_tr_t['step'].min()} .. {X_tr_t['step'].max()}]")
print(f"Time-based Test Step Range:  [{X_te_t['step'].min()} .. {X_te_t['step'].max()}]")
print(f"Train Fraud Cases: {y_tr_t.sum():,} | Test Fraud Cases: {y_te_t.sum():,}")

## 7. Numerical Feature Scaling: Why `RobustScaler`?
`StandardScaler` uses the formula $z = \frac{x - \mu}{\sigma}$. Because transaction amounts contain extreme outliers (e.g. transfers of \$10,000,000+), the mean and variance become distorted.

`RobustScaler` uses the **median** and **Interquartile Range (IQR)**:

$$x_{\text{scaled}} = \frac{x - Q_2}{Q_3 - Q_1}$$

This ensures extreme outlier amounts do not compress normal transaction values to zero.

In [ ]:
# Fit our production-grade FraudDataPreprocessor strictly on X_train
preprocessor = FraudDataPreprocessor(
    engineer_features=True,
    scale_features=True,
    drop_identifiers=True,
    drop_leakage_features=True
)

print("Fitting preprocessor on X_train...")
X_train_proc = preprocessor.fit_transform(X_train)

print("Transforming X_test using learned parameters...")
X_test_proc = preprocessor.transform(X_test)

print(f"\nProcessed Train Matrix: {X_train_proc.shape[0]:,} rows x {X_train_proc.shape[1]} features")
print(f"Processed Test Matrix:  {X_test_proc.shape[0]:,} rows x {X_test_proc.shape[1]} features")
print("\nOutput Features in Deterministic Order:")
for i, col in enumerate(preprocessor.get_feature_names_out(), 1):
    print(f"  {i:02d}. {col}")

## 8. Verifying Processed Feature Distributions

In [ ]:
display(X_train_proc.head())

# Visualizing transformed features
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=X_train_proc[['amount', 'oldbalanceOrg', 'newbalanceOrig']], ax=axes[0])
axes[0].set_title("Robustly Scaled Balances & Amount", fontweight="bold")

sns.boxplot(data=X_train_proc[['errorBalanceOrig', 'errorBalanceDest']], ax=axes[1])
axes[1].set_title("Engineered Balance Errors (Robustly Scaled)", fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Serialization & Artifact Persistence
We save our fitted preprocessor pipeline to `models/preprocessor.joblib`. This serialized object will be loaded by FastAPI and Streamlit to transform real-time inference transactions.

In [ ]:
pipeline_path = MODELS_DIR / "preprocessor.joblib"
preprocessor.save(pipeline_path)
print(f"Fitted pipeline saved to: {pipeline_path.resolve()}")

# Verify reloadability
loaded_prep = FraudDataPreprocessor.load(pipeline_path)
sample_inference = X_test.iloc[0:1]
assert loaded_prep.transform(sample_inference).equals(preprocessor.transform(sample_inference))
print("Verified: Serialized pipeline reloads and produces identical outputs!")

## 10. Phase 2 Summary & Decisions

| Preprocessing Step | Implementation | Reason / Technical Decision |
| :--- | :--- | :--- |
| **Data Splitting** | Stratified `train_test_split` | Preserves the rare 0.13% fraud proportion in both splits; prevents data leakage. |
| **High-Cardinality IDs** | Dropped `nameOrig` and `nameDest` | Prevents dimensionality explosion and model memorization/overfitting. |
| **Feature Engineering** | `errorBalanceOrig`, `errorBalanceDest` | Exploits accounting discrepancies common in fraudulent drain transactions. |
| **Entity Classification** | `isDestMerchant` derived from ID prefix | Flags whether transfer target is a merchant vs customer account. |
| **Leakage Guard** | Dropped `isFlaggedFraud` | Prevents target contamination from simulation heuristics. |
| **Categorical Encoding** | One-Hot Encoding with `handle_unknown='ignore'` | Safely handles new/unseen transaction categories during inference. |
| **Numerical Scaling** | `RobustScaler` (median & IQR) | Resilient against massive transaction amount outliers. |
| **Pipeline Architecture** | Scikit-learn Transformer Class | Reusable across Training, FastAPI REST API, and Frontend dashboard. |

---
**Phase 2 Complete:** The preprocessing pipeline is fully implemented, verified, leak-free, and serialized for modeling.